## 표준유역코드(codeWatershed) labelling

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
obsAll = pd.read_csv('관측소목록_수위자료_한강.csv', encoding='utf8', sep=',', )
obsHrcf = pd.read_csv('관측소목록_한강홍수통제소.csv', encoding='utf8', sep=',',)

In [ ]:
obsAll.head()

,대권역,관측소 명,운영 여부,관측 방법,관측소 코드,표준유역 코드,관할 기관
0,한강,평창군(송정교),운영,T/M,1001602,100108,기후에너지환경부
1,한강,태백시(무사교),운영,T/M,1001603,100101,한국수자원공사
2,한강,송천1교,운영,T/M,1001605,100105,한국수력원자력
3,한강,삼척시(번천교),운영,T/M,1001607,100101,한국수자원공사
4,한강,삼척시(광동교),운영,T/M,1001613,100102,한국수자원공사


In [ ]:
obsHrcf.head()

,지점,관할기관,3시간전 수위(m),수위 (m),유량,해발수위 (El.m)
0,가평군(가평교),기후에너지환경부,0.76,0.76,3.91,51.62
1,가평군(대보교),기후에너지환경부,2.66,2.66,1.81,102.19
2,가평군(선촌2교),기후에너지환경부,2.32,2.31,-,52.88
3,가평군(신상교),기후에너지환경부,3.49,3.48,-,137.63
4,가평군(신청평대교),기후에너지환경부,1.34,1.32,-,23.93


In [ ]:
print(obsAll.columns)
print(obsHrcf.columns)

Index(['대권역', '관측소 명', '운영 여부', '관측 방법', '관측소 코드', '표준유역 코드', '관할 기관'], dtype='object')
Index(['지점', '관할기관', '3시간전 수위(m)', '수위 (m)', '유량', '해발수위 (El.m)'], dtype='object')


In [ ]:
columns_all = list(obsAll.columns.copy())
columns_all[1] = '지점'
obsAll.columns = columns_all

In [ ]:
obsChecked = pd.merge(obsHrcf, obsAll, how='inner', on='지점')
obsMerged = pd.merge(obsHrcf, obsAll, how='left', on='지점')

In [ ]:
obsMerged.head()

,지점,관할기관,3시간전 수위(m),수위 (m),유량,해발수위 (El.m),대권역,운영 여부,관측 방법,관측소 코드,표준유역 코드,관할 기관
0,가평군(가평교),기후에너지환경부,0.76,0.76,3.91,51.62,한강,운영,T/M,1013655.0,101306.0,기후에너지환경부
1,가평군(대보교),기후에너지환경부,2.66,2.66,1.81,102.19,한강,운영,T/M,1015620.0,101504.0,기후에너지환경부
2,가평군(선촌2교),기후에너지환경부,2.32,2.31,-,52.88,한강,운영,T/M,1015636.0,101501.0,기후에너지환경부
3,가평군(신상교),기후에너지환경부,3.49,3.48,-,137.63,한강,운영,T/M,1015611.0,101503.0,기후에너지환경부
4,가평군(신청평대교),기후에너지환경부,1.34,1.32,-,23.93,한강,운영,T/M,1015645.0,101506.0,기후에너지환경부


In [ ]:
obsMerged.tail()

,지점,관할기관,3시간전 수위(m),수위 (m),유량,해발수위 (El.m),대권역,운영 여부,관측 방법,관측소 코드,표준유역 코드,관할 기관
286,횡성군(안흥교),기후에너지환경부,1.17,1.16,3.88,415.46,한강,운영,T/M,1002675.0,100208.0,기후에너지환경부
287,횡성군(오산교),기후에너지환경부,1.89,1.89,1.52,120.28,한강,운영,T/M,1006630.0,100603.0,기후에너지환경부
288,횡성군(전천교),기후에너지환경부,0.94,0.94,0.02,108.82,한강,운영,T/M,1006660.0,100604.0,기후에너지환경부
289,횡성군(청곡교),기후에너지환경부,3.48,3.47,-,149.16,한강,운영,T/M,1006628.0,100603.0,기후에너지환경부
290,횡성군(횡성교),기후에너지환경부,1.07,1.07,8.19,107.12,한강,운영,T/M,1006650.0,100605.0,기후에너지환경부


In [ ]:
uniqObsHrcf = set(obsHrcf['지점']) - set(obsChecked['지점'])
uniqObsAll = set(obsAll['지점']) - set(obsChecked['지점'])

In [ ]:
# WAMIS 자료로 처리 안 된 한강홍수통제소 관측소 확인

print(f"WAMIS 파일에 없는 HRCF 관측소: {len(list(uniqObsHrcf))}")
print('\n'.join(list(uniqObsHrcf)))

WAMIS 파일에 없는 HRCF 관측소: 5
강릉시(난곡교)
강릉시(우정교)
영월군(주채교)
속초시(응골교)
고성군(간촌교)


In [ ]:
# 해당 관측소 -> 좌표 OR 하천 & GIS 시스템 이용 -> 직접 표준유역코드 삽입
  # 관측소 좌표 검색 -> 표준유역코드 좌표 검색
  # 관측소 하천 검색 -> 표준유역코드 하천 검색
    # 관측소 - 표준유역코드 매핑

import collections

obsNotfound = collections.defaultdict(dict)

# 하천명 네이버 검색 -> 기후에너지환경부 한강홍수통제소_표준유역코드_240919.csv 에서 하천 - 표준유역코드 매핑
# 영월군(주채교)와 고성군(간촌교)는 Gemini 검색 -> 이하 동일

missing_obs_data = [
    {'지점': '강릉시(난곡교)', '하천': '경포천', '표준유역코드': '320309'},
      # 기후부 자료에 전라도 경포천 코드만 있음 # 대권역 코드가 10-13 여야 함.
    {'지점': '강릉시(우정교)', '하천': '연곡천', '표준유역코드': '130201'},
    {'지점': '영월군(주채교)', '하천': '옥동천상류', '표준유역코드': '100302'},
    {'지점': '속초시(응골교)', '하천': '청초천', '표준유역코드': '130105'},
    {'지점': '고성군(간촌교)', '하천': '북천', '표준유역코드': '101103'}
]

for obs in missing_obs_data:
    obsNotfound[obs['지점']] = obs['표준유역코드']

In [ ]:
obsMerged.columns

Index(['지점', '관할기관', '3시간전 수위(m)', '수위 (m)', '유량', '해발수위 (El.m)', '대권역',
       '운영 여부', '관측 방법', '관측소 코드', '표준유역 코드', '관할 기관'],
      dtype='object')

In [ ]:
obsNotfound.items()

dict_items([('강릉시(난곡교)', '320309'), ('강릉시(우정교)', '130201'), ('영월군(주채교)', '100302'), ('속초시(응골교)', '130105'), ('고성군(간촌교)', '101103')])

In [ ]:
for obs, codeWatershed in obsNotfound.items():
  obsMerged.loc[obsMerged['지점']==obs, '표준유역 코드'] = int(codeWatershed)

In [ ]:
obsMerged[obsMerged['지점']=='강릉시(난곡교)']

,지점,관할기관,3시간전 수위(m),수위 (m),유량,해발수위 (El.m),대권역,운영 여부,관측 방법,관측소 코드,표준유역 코드,관할 기관
10,강릉시(난곡교),기후에너지환경부,1.96,1.96,-,4.96,NaN,NaN,NaN,NaN,320309,NaN


In [ ]:
obsMerged.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 291 entries, 0 to 290
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   지점           291 non-null    object 
 1   관할기관         291 non-null    object 
 2   3시간전 수위(m)   291 non-null    float64
 3   수위 (m)       291 non-null    float64
 4   유량           291 non-null    object 
 5   해발수위 (El.m)  291 non-null    object 
 6   대권역          286 non-null    object 
 7   운영 여부        286 non-null    object 
 8   관측 방법        286 non-null    object 
 9   관측소 코드       286 non-null    float64
 10  표준유역 코드      291 non-null    object 
 11  관할 기관        286 non-null    object 
dtypes: float64(3), object(9)
memory usage: 27.4+ KB


In [ ]:
obsMerged['표준유역 코드'] = obsMerged['표준유역 코드'].astype(int)

In [ ]:
obsMerged.to_csv('/content/drive/MyDrive/FloodAX/metadata_outputs/obsCodeWatershed.csv', encoding='utf8')

In [ ]:
obsChecked.shape

(286, 12)

**한강홍수통제소 관측소에 표준유역코드 labelling**
```obsMerged```
```obsCodeWatershed.csv``` \
```output_dir``` = "/content/drive/MyDrive/FloodAX"

In [ ]:
obsMerged = pd.read_csv('/content/drive/MyDrive/FloodAX/metadata_outputs/obsCodeWatershed.csv', sep=',', encoding='utf8')

In [ ]:
# 표준유역코드 Sanity Check
set(obsMerged['표준유역 코드'])

{100102,
 100103,
 100106,
 100107,
 100108,
 100109,
 100110,
 100111,
 100112,
 100113,
 100114,
 100115,
 100116,
 100117,
 100201,
 100202,
 100203,
 100204,
 100205,
 100206,
 100207,
 100208,
 100209,
 100210,
 100212,
 100213,
 100302,
 100303,
 100304,
 100305,
 100306,
 100307,
 100308,
 100309,
 100310,
 100313,
 100314,
 100315,
 100316,
 100317,
 100318,
 100402,
 100403,
 100404,
 100407,
 100408,
 100409,
 100410,
 100412,
 100413,
 100414,
 100502,
 100503,
 100504,
 100505,
 100603,
 100604,
 100605,
 100606,
 100607,
 100608,
 100609,
 100610,
 100701,
 100702,
 100703,
 100704,
 100705,
 100706,
 100707,
 100708,
 100709,
 100710,
 100711,
 100712,
 100713,
 100714,
 100715,
 100716,
 100717,
 100718,
 100719,
 101004,
 101006,
 101008,
 101010,
 101011,
 101012,
 101102,
 101103,
 101104,
 101201,
 101202,
 101203,
 101204,
 101205,
 101208,
 101301,
 101302,
 101303,
 101304,
 101306,
 101402,
 101403,
 101404,
 101405,
 101406,
 101407,
 101408,
 101409,
 101410,
 

In [ ]:
# HRCF에 없는 WAMIS 파일 관측소 확인

print(f"HRCF에 없는 WAMIS 파일의 관측소: {len(list(uniqObsAll))}")
print('\n'.join(list(uniqObsAll)))

HRCF에 없는 WAMIS 파일의 관측소: 78
태백시(무사교)
용당
인제군(어두원교)
소수
원남
연천군(한여울교)
약수교
후영교
삼척시(번천교)
아라서해갑문(내)
인제군(원통리)
홍천강(성포교)
포천시(영로대교)
대평
우화교
구용산
포천시(대회산교)
영월군(삼옥교)
인제군(살구미교)
명서교
아산만(내)
삼척시(광동교)
화천군(평화나래교)
고성교
인제군(원대교)
아라서해갑실(남)
토교
동해시(원평교)
이동
오봉
오탄교
궁신교
영월군(판운교)
귤현보(상)
경안교
인제군(도리촌교)
화천군(화천대교)
충주시(향산리)
고삼
송천1교
인제군(양구교)
아라한강갑문(외)
아라서해갑실(북)
용신교
한덕교
춘천시(소양6교)
화천군(오작교)
충주본댐우안
팔봉교
아라서해갑문(외)
연천군(고탄교)
단양군(북벽교)
양구군(각시교)
아라한강갑문(내)
계양대교
철원군(백마교)
철원군(군탄교)
하리교
아라한강갑실
궁촌
제천시(물태리)
횡성군(율동리)
여주저류지
벌말교
영월군(연당교)
남전교
삼척시(갈밭교)
연천군(한탄강댐)
횡성군(포동2교)
반계
아산만(외)
상진교
목동교
산장제1교
귤현보(하)
금사
박촌1교
금광


In [ ]:
set(obsAll[obsAll['지점'].isin(uniqObsAll)]['관할 기관'])

{'기후에너지환경부', '한국농어촌공사', '한국수력원자력', '한국수자원공사'}

In [ ]:
obsAll[(obsAll['지점'].isin(uniqObsAll)) & (obsAll['관할 기관'] == "기후에너지환경부")]

,대권역,지점,운영 여부,관측 방법,관측소 코드,표준유역 코드,관할 기관
247,한강,구용산,운영,보통,1018685,101711,기후에너지환경부


기후에너지환경부 소관이고 대권역이 한강인데, 한강홍수통제소에는 데이터가 없는 관측소 -> **구용산 / 1018685** -> 직접 확인 필요

## 하천차수(numStream) labelling
`numStream` "WAMIS_하천차수.csv" 이용

---

`하천차수`: 하천의 분기 정도를 구분하기 위하여 1차, 2차 등으로 부여하는 수치 \
`차수`: 하천의 굵기. 한강 본류 같은 큰 강은 6~7차. 숫자가 작을수록 개울이고 클수록 하류의 큰 강


---

**1차 하천**: 손가락 끝처럼 더이상 갈라지지 않는 가장 위쪽의 실개천 \

**치수 상승 조건**: 동일 차수의 하천 두 개가 만나면 한 차수 증가



In [ ]:
numStream = pd.read_csv('/content/drive/MyDrive/FloodAX/metadata/WAMIS_하천차수.csv', encoding='utf8', sep=',')

In [ ]:
numStream.head()

,유역코드,유역명,1차,2차,3차,4차,5차,6차,7차,8차
0,100101,광동댐,392,85,19,5,2,1,-,-
1,100102,광동댐하류,660,140,37,6,2,1,-,-
2,100103,임계천,525,129,29,5,1,1,-,-
3,100104,골지천중류,405,87,18,4,2,1,-,-
4,100105,도암댐,121,34,8,3,1,-,-,-


In [ ]:
numStream.columns

Index(['유역코드', '유역명', '1차', '2차', '3차', '4차', '5차', '6차', '7차', '8차'], dtype='object')

In [ ]:
obsMerged.head()

,Unnamed: 0,지점,관할기관,3시간전 수위(m),수위 (m),유량,해발수위 (El.m),대권역,운영 여부,관측 방법,관측소 코드,표준유역 코드,관할 기관
0,0,가평군(가평교),기후에너지환경부,0.76,0.76,3.91,51.62,한강,운영,T/M,1013655.0,101306,기후에너지환경부
1,1,가평군(대보교),기후에너지환경부,2.66,2.66,1.81,102.19,한강,운영,T/M,1015620.0,101504,기후에너지환경부
2,2,가평군(선촌2교),기후에너지환경부,2.32,2.31,-,52.88,한강,운영,T/M,1015636.0,101501,기후에너지환경부
3,3,가평군(신상교),기후에너지환경부,3.49,3.48,-,137.63,한강,운영,T/M,1015611.0,101503,기후에너지환경부
4,4,가평군(신청평대교),기후에너지환경부,1.34,1.32,-,23.93,한강,운영,T/M,1015645.0,101506,기후에너지환경부


In [ ]:
obsMerged.columns

Index(['Unnamed: 0', '지점', '관할기관', '3시간전 수위(m)', '수위 (m)', '유량', '해발수위 (El.m)',
       '대권역', '운영 여부', '관측 방법', '관측소 코드', '표준유역 코드', '관할 기관'],
      dtype='object')

In [ ]:
# 하천차수 정보 없는 한강홍수통제소 관측소
uniqObsHrcf = set(obsMerged.loc[:, '표준유역 코드']) - set(numStream.loc[:, '유역코드'])

In [ ]:
uniqObsHrcf

{320309}

In [ ]:
# 컬럼명 통일 후 merge
obsMergedEngNorm = ['indexObs', 'korObs', 'korInst', 'waterLevel3hr', 'waterLevel', 'waterFlux', 'waterElevation', 'sphereLarge', 'inOperation', 'methodObs', 'codeObs', 'codeWatershed', 'korInstduplicate']
numStreamEngNorm = ['codeWatershed', 'korStream', '1차', '2차', '3차', '4차', '5차', '6차', '7차', '8차']

In [ ]:
obsMerged.columns = obsMergedEngNorm
numStream.columns = numStreamEngNorm

In [ ]:
obsMerged.head()

,indexObs,korObs,korInst,waterLevel3hr,waterLevel,waterFlux,waterElevation,sphereLarge,inOperation,methodObs,codeObs,codeWatershed,korInstduplicate
0,0,가평군(가평교),기후에너지환경부,0.76,0.76,3.91,51.62,한강,운영,T/M,1013655.0,101306,기후에너지환경부
1,1,가평군(대보교),기후에너지환경부,2.66,2.66,1.81,102.19,한강,운영,T/M,1015620.0,101504,기후에너지환경부
2,2,가평군(선촌2교),기후에너지환경부,2.32,2.31,-,52.88,한강,운영,T/M,1015636.0,101501,기후에너지환경부
3,3,가평군(신상교),기후에너지환경부,3.49,3.48,-,137.63,한강,운영,T/M,1015611.0,101503,기후에너지환경부
4,4,가평군(신청평대교),기후에너지환경부,1.34,1.32,-,23.93,한강,운영,T/M,1015645.0,101506,기후에너지환경부


In [ ]:
obsMerged = obsMerged.loc[:, ['korObs', 'korInst', 'waterElevation', 'sphereLarge', 'inOperation', 'methodObs', 'codeObs', 'codeWatershed']]
obsMergedFinal = pd.merge(obsMerged, numStream, how='left', on='codeWatershed')

In [ ]:
obsMergedFinal.head()

,korObs,korInst,waterElevation,sphereLarge,inOperation,methodObs,codeObs,codeWatershed,korStream,1차,2차,3차,4차,5차,6차,7차,8차
0,가평군(가평교),기후에너지환경부,51.62,한강,운영,T/M,1013655.0,101306,가평천하류,143.0,38,8,3,2,1,-,-
1,가평군(대보교),기후에너지환경부,102.19,한강,운영,T/M,1015620.0,101504,조종천하류,86.0,20,5,2,1,-,-,-
2,가평군(선촌2교),기후에너지환경부,52.88,한강,운영,T/M,1015636.0,101501,미원천,75.0,19,5,2,2,-,-,-
3,가평군(신상교),기후에너지환경부,137.63,한강,운영,T/M,1015611.0,101503,조종천상류,77.0,25,5,2,1,-,-,-
4,가평군(신청평대교),기후에너지환경부,23.93,한강,운영,T/M,1015645.0,101506,청평댐하류,139.0,30,6,1,1,-,1,-


In [ ]:
# 결측치 제대로 표시
obsMergedFinal = obsMergedFinal.map(lambda x: np.nan if x == '-' else x)

In [ ]:
obsMergedFinal.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 291 entries, 0 to 290
Data columns (total 17 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   korObs          291 non-null    object 
 1   korInst         291 non-null    object 
 2   waterElevation  289 non-null    object 
 3   sphereLarge     286 non-null    object 
 4   inOperation     286 non-null    object 
 5   methodObs       286 non-null    object 
 6   codeObs         286 non-null    float64
 7   codeWatershed   291 non-null    int64  
 8   korStream       290 non-null    object 
 9   1차              290 non-null    float64
 10  2차              289 non-null    object 
 11  3차              289 non-null    object 
 12  4차              278 non-null    object 
 13  5차              227 non-null    object 
 14  6차              135 non-null    object 
 15  7차              50 non-null     object 
 16  8차              30 non-null     object 
dtypes: float64(2), int64(1), object(14)

In [ ]:
for column in obsMergedFinal.columns:
  print(column, sum(obsMergedFinal.loc[:, column].isna()))

korObs 0
korInst 0
waterElevation 2
sphereLarge 5
inOperation 5
methodObs 5
codeObs 5
codeWatershed 0
korStream 1
1차 1
2차 2
3차 2
4차 13
5차 64
6차 156
7차 241
8차 261


5개 관측소 대해서만 phereLarge, inOperation, methodObs, codeObs, ... 등 없는 이유-> **WAMIS 파일에 해당 obs 데이터 없어서**

In [ ]:
na_rows = obsMergedFinal[obsMergedFinal.isna().any(axis=1)]
display(na_rows)

,korObs,korInst,waterElevation,sphereLarge,inOperation,methodObs,codeObs,codeWatershed,korStream,1차,2차,3차,4차,5차,6차,7차,8차
0,가평군(가평교),기후에너지환경부,51.62,한강,운영,T/M,1013655.0,101306,가평천하류,143.0,38,8,3,2,1,NaN,NaN
1,가평군(대보교),기후에너지환경부,102.19,한강,운영,T/M,1015620.0,101504,조종천하류,86.0,20,5,2,1,NaN,NaN,NaN
2,가평군(선촌2교),기후에너지환경부,52.88,한강,운영,T/M,1015636.0,101501,미원천,75.0,19,5,2,2,NaN,NaN,NaN
3,가평군(신상교),기후에너지환경부,137.63,한강,운영,T/M,1015611.0,101503,조종천상류,77.0,25,5,2,1,NaN,NaN,NaN
4,가평군(신청평대교),기후에너지환경부,23.93,한강,운영,T/M,1015645.0,101506,청평댐하류,139.0,30,6,1,1,NaN,1,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
286,횡성군(안흥교),기후에너지환경부,415.46,한강,운영,T/M,1002675.0,100208,주천강상류,406.0,88,15,5,1,NaN,NaN,NaN
287,횡성군(오산교),기후에너지환경부,120.28,한강,운영,T/M,1006630.0,100603,금계천,161.0,35,10,1,NaN,NaN,NaN,NaN
288,횡성군(전천교),기후에너지환경부,108.82,한강,운영,T/M,1006660.0,100604,전천,446.0,109,23,5,1,NaN,NaN,NaN
289,횡성군(청곡교),기후에너지환경부,149.16,한강,운영,T/M,1006628.0,100603,금계천,161.0,35,10,1,NaN,NaN,NaN,NaN


In [ ]:
obsMergedFinal.to_csv('/content/drive/MyDrive/FloodAX/metadata_outputs/obsCodeFinal.csv', encoding='utf8')

## 계획홍수위/수량(designFloodLevel/designFloodCharge) labelling

In [ ]:
obsDesign = pd.read_csv("/content/drive/MyDrive/FloodAX/metadata/한강홍수통제소_계획홍수위수량.csv", sep=',')

In [ ]:
obsDesign.head(20)


,수계,홍수,특보,특보.1,홍수.1,Unnamed: 5,Unnamed: 6,Unnamed: 7,계획홍수위,Unnamed: 9,계획,수위표,제방고,Unnamed: 13,Unnamed: 14,Unnamed: 15,하폭
0,NaN,특보,지점,지점,특보 수위,NaN,NaN,NaN,NaN,NaN,홍수량,영점,NaN,NaN,NaN,NaN,(m)
1,NaN,지점명,위치,및,NaN,NaN,NaN,NaN,NaN,NaN,(㎥/sec),표고,NaN,NaN,NaN,NaN,NaN
2,NaN,(하천명),NaN,기준,NaN,NaN,NaN,NaN,NaN,NaN,NaN,(El.m),NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,수위,주의보,NaN,경 보,NaN,NaN,NaN,NaN,NaN,좌안,NaN,우안,NaN,NaN
4,NaN,NaN,NaN,고시일,수위표,해발,수위표,해발,수위표,해발,NaN,NaN,수위표,해발,수위표,해발,NaN
5,NaN,NaN,NaN,NaN,기준,기준,기준,기준,기준,기준,NaN,NaN,기준,기준,기준,기준,NaN
6,NaN,NaN,NaN,NaN,(m),(El.m),(m),(El.m),(m),(El.m),NaN,NaN,(m),(El.m),(m),(El.m),NaN
7,한강,서울시,서울시,00.04.15,8.5,10.57,10.5,12.57,12.8,14.87,"37,000",2.07,15.93,18,16.43,18.5,"1,013"
8,NaN,(한강대교),한강대교,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,NaN,(한강),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
obsDesign.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 295 entries, 0 to 294
Data columns (total 17 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   수계           55 non-null     object
 1   홍수           195 non-null    object
 2   특보           131 non-null    object
 3   특보.1         69 non-null     object
 4   홍수.1         69 non-null     object
 5   Unnamed: 5   67 non-null     object
 6   Unnamed: 6   68 non-null     object
 7   Unnamed: 7   67 non-null     object
 8   계획홍수위        67 non-null     object
 9   Unnamed: 9   67 non-null     object
 10  계획           66 non-null     object
 11  수위표          67 non-null     object
 12  제방고          68 non-null     object
 13  Unnamed: 13  67 non-null     object
 14  Unnamed: 14  68 non-null     object
 15  Unnamed: 15  67 non-null     object
 16  하폭           65 non-null     object
dtypes: object(17)
memory usage: 39.3+ KB


In [ ]:
obsDesign.columns

Index(['수계', '홍수', '특보', '특보.1', '홍수.1', 'Unnamed: 5', 'Unnamed: 6',
       'Unnamed: 7', '계획홍수위', 'Unnamed: 9', '계획', '수위표', '제방고', 'Unnamed: 13',
       'Unnamed: 14', 'Unnamed: 15', '하폭'],
      dtype='object')

In [ ]:
obsDesignEngNorm = ['korStream', 'korFloodAlert(korStream)', 'locFloodAlert', 'dateNotice',
                    'aFLAdvisory(m)', 'aFLAdvisory(El.m)', 'aFLAlert(m)', 'aFLAlert(El.m)',
                    'designFloodLevel(m)', 'designFloodLevel(El.m)', 'designFloodCharge(m3/sec)',
                    '(El.m)',
                    'heightEmbarkmentLeft(m)', 'heightEmbarkmentLeft(El.m)', 'heightEmbarkmentRight(m)', 'heightEmbarkmentRight(El.m)',
                    'widthRiver(m)' ]
obsDesign.columns = obsDesignEngNorm

In [ ]:
obsDesign = obsDesign.loc[7:, ]

In [ ]:
obsDesign.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 288 entries, 7 to 294
Data columns (total 17 columns):
 #   Column                       Non-Null Count  Dtype 
---  ------                       --------------  ----- 
 0   korStream                    55 non-null     object
 1   korFloodAlert(korStream)     192 non-null    object
 2   locFloodAlert                129 non-null    object
 3   dateNotice                   64 non-null     object
 4   aFLAdvisory(m)               64 non-null     object
 5   aFLAdvisory(El.m)            64 non-null     object
 6   aFLAlert(m)                  64 non-null     object
 7   aFLAlert(El.m)               64 non-null     object
 8   designFloodLevel(m)          64 non-null     object
 9   designFloodLevel(El.m)       64 non-null     object
 10  designFloodCharge(m3/sec)    64 non-null     object
 11  (El.m)                       64 non-null     object
 12  heightEmbarkmentLeft(m)      64 non-null     object
 13  heightEmbarkmentLeft(El.m)   64 non

In [ ]:
# korStream은 앞에 있는 값으로 채우기 (ffill)
obsDesign['korStream'] = obsDesign['korStream'].ffill()

In [ ]:
obsDesign.head(20)

,korStream,korFloodAlert(korStream),locFloodAlert,dateNotice,aFLAdvisory(m),aFLAdvisory(El.m),aFLAlert(m),aFLAlert(El.m),designFloodLevel(m),designFloodLevel(El.m),designFloodCharge(m3/sec),(El.m),heightEmbarkmentLeft(m),heightEmbarkmentLeft(El.m),heightEmbarkmentRight(m),heightEmbarkmentRight(El.m),widthRiver(m)
7,한강,서울시,서울시,00.04.15,8.5,10.57,10.5,12.57,12.8,14.87,"37,000",2.07,15.93,18,16.43,18.5,"1,013"
8,한강,(한강대교),한강대교,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,한강,(한강),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
10,한강,정선군,정선군,24.04.23,5.5,337.56,6.5,338.56,7.56,339.62,"3,669",332.06,8.88,340.94,8.88,340.94,-
11,한강,(남평대교),남평대교,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
12,한강,(한강),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
13,한강,여주시,여주시,12.05.15,6,38.53,8,40.53,9.19,41.72,"16,070",32.53,14.6,47.13,14.59,47.12,502
14,한강,(여주대교),여주대교,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
15,한강,(한강),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
16,한강,정선군,정선군,21.12.28,6.7,301.99,8.6,303.89,10.5,305.79,"4,934",295.29,15.52,310.81,15.5,310.79,340


In [ ]:
idxColSave = list(range(obsDesign.shape[1]))
idxColSave

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16]

In [ ]:
idxColSave.remove(2)

In [ ]:
idxColSave

[0, 1, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16]

In [ ]:
# locFloodAlert 삭제하고
obsDesign = obsDesign.iloc[:, idxColSave]

In [ ]:
# korFloodAlert(korStream)에서 (korStream) 삭제

idxRowSave1 = list(range(0, obsDesign.shape[0],3 ))
idxRowSave2 = list(range(1, obsDesign.shape[0], 3))
idxRowSave = idxRowSave1 + idxRowSave2

In [ ]:
idxRowSave.sort()

In [ ]:
idxRowSave

[0,
 1,
 3,
 4,
 6,
 7,
 9,
 10,
 12,
 13,
 15,
 16,
 18,
 19,
 21,
 22,
 24,
 25,
 27,
 28,
 30,
 31,
 33,
 34,
 36,
 37,
 39,
 40,
 42,
 43,
 45,
 46,
 48,
 49,
 51,
 52,
 54,
 55,
 57,
 58,
 60,
 61,
 63,
 64,
 66,
 67,
 69,
 70,
 72,
 73,
 75,
 76,
 78,
 79,
 81,
 82,
 84,
 85,
 87,
 88,
 90,
 91,
 93,
 94,
 96,
 97,
 99,
 100,
 102,
 103,
 105,
 106,
 108,
 109,
 111,
 112,
 114,
 115,
 117,
 118,
 120,
 121,
 123,
 124,
 126,
 127,
 129,
 130,
 132,
 133,
 135,
 136,
 138,
 139,
 141,
 142,
 144,
 145,
 147,
 148,
 150,
 151,
 153,
 154,
 156,
 157,
 159,
 160,
 162,
 163,
 165,
 166,
 168,
 169,
 171,
 172,
 174,
 175,
 177,
 178,
 180,
 181,
 183,
 184,
 186,
 187,
 189,
 190,
 192,
 193,
 195,
 196,
 198,
 199,
 201,
 202,
 204,
 205,
 207,
 208,
 210,
 211,
 213,
 214,
 216,
 217,
 219,
 220,
 222,
 223,
 225,
 226,
 228,
 229,
 231,
 232,
 234,
 235,
 237,
 238,
 240,
 241,
 243,
 244,
 246,
 247,
 249,
 250,
 252,
 253,
 255,
 256,
 258,
 259,
 261,
 262,
 264,
 265,
 267,


In [ ]:
obsDesign = obsDesign.iloc[idxRowSave, :]

In [ ]:
obsDesign.head()

,korStream,korFloodAlert(korStream),dateNotice,aFLAdvisory(m),aFLAdvisory(El.m),aFLAlert(m),aFLAlert(El.m),designFloodLevel(m),designFloodLevel(El.m),designFloodCharge(m3/sec),(El.m),heightEmbarkmentLeft(m),heightEmbarkmentLeft(El.m),heightEmbarkmentRight(m),heightEmbarkmentRight(El.m),widthRiver(m)
7,한강,서울시,00.04.15,8.5,10.57,10.5,12.57,12.8,14.87,"37,000",2.07,15.93,18,16.43,18.5,"1,013"
8,한강,(한강대교),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
10,한강,정선군,24.04.23,5.5,337.56,6.5,338.56,7.56,339.62,"3,669",332.06,8.88,340.94,8.88,340.94,-
11,한강,(남평대교),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
13,한강,여주시,12.05.15,6,38.53,8,40.53,9.19,41.72,"16,070",32.53,14.6,47.13,14.59,47.12,502


In [ ]:
# korFloodAlert 두 행씩 합치기
  # 강릉시 \ (회산교) \ (강릉남대천) -> 강릉시 \ (회산교) -> 강릉시 (회산교)

# Get the column index for 'korFloodAlert(korStream)'
col_name = 'korFloodAlert(korStream)'
col_idx = obsDesign.columns.get_loc(col_name)

for i in range(0, obsDesign.shape[0], 2):
  # Ensure the DataFrame has enough rows for i+1
  if i + 1 < obsDesign.shape[0]:
    # Convert values to string before concatenating to handle potential non-string types or NaN
    val1 = str(obsDesign.iloc[i, col_idx])
    val2 = str(obsDesign.iloc[i+1, col_idx])

    # Concatenate the values and update the even-indexed row
    # Added a space between concatenated strings for better readability
    obsDesign.iloc[i, col_idx] = val1 + val2

In [ ]:
obsDesign.head(10)

,korStream,korFloodAlert(korStream),dateNotice,aFLAdvisory(m),aFLAdvisory(El.m),aFLAlert(m),aFLAlert(El.m),designFloodLevel(m),designFloodLevel(El.m),designFloodCharge(m3/sec),(El.m),heightEmbarkmentLeft(m),heightEmbarkmentLeft(El.m),heightEmbarkmentRight(m),heightEmbarkmentRight(El.m),widthRiver(m)
7,한강,서울시(한강대교),00.04.15,8.5,10.57,10.5,12.57,12.8,14.87,"37,000",2.07,15.93,18,16.43,18.5,"1,013"
8,한강,(한강대교),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
10,한강,정선군(남평대교),24.04.23,5.5,337.56,6.5,338.56,7.56,339.62,"3,669",332.06,8.88,340.94,8.88,340.94,-
11,한강,(남평대교),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
13,한강,여주시(여주대교),12.05.15,6,38.53,8,40.53,9.19,41.72,"16,070",32.53,14.6,47.13,14.59,47.12,502
14,한강,(여주대교),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
16,한강,정선군(정선제1교),21.12.28,6.7,301.99,8.6,303.89,10.5,305.79,"4,934",295.29,15.52,310.81,15.5,310.79,340
17,한강,(정선제1교),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
19,한강,영월군(영월대교),07.04.05,6.6,189.81,8.5,191.71,10.5,193.71,"5,656",183.21,13.8,197.01,13.8,197.01,260
20,한강,(영월대교),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# 결측치 있는 행 삭제
obsDesign = obsDesign.dropna()
  # widthRiver은 결측치 표현인 "-" 그대로 냅둠.

In [ ]:
obsDesign

,korStream,korFloodAlert(korStream),dateNotice,aFLAdvisory(m),aFLAdvisory(El.m),aFLAlert(m),aFLAlert(El.m),designFloodLevel(m),designFloodLevel(El.m),designFloodCharge(m3/sec),(El.m),heightEmbarkmentLeft(m),heightEmbarkmentLeft(El.m),heightEmbarkmentRight(m),heightEmbarkmentRight(El.m),widthRiver(m)
7,한강,서울시(한강대교),00.04.15,8.5,10.57,10.5,12.57,12.8,14.87,"37,000",2.07,15.93,18,16.43,18.5,"1,013"
10,한강,정선군(남평대교),24.04.23,5.5,337.56,6.5,338.56,7.56,339.62,"3,669",332.06,8.88,340.94,8.88,340.94,-
13,한강,여주시(여주대교),12.05.15,6,38.53,8,40.53,9.19,41.72,"16,070",32.53,14.6,47.13,14.59,47.12,502
16,한강,정선군(정선제1교),21.12.28,6.7,301.99,8.6,303.89,10.5,305.79,"4,934",295.29,15.52,310.81,15.5,310.79,340
19,한강,영월군(영월대교),07.04.05,6.6,189.81,8.5,191.71,10.5,193.71,"5,656",183.21,13.8,197.01,13.8,197.01,260
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
184,진위천,평택시(진위1교),24.04.23,4.6,9.93,5.8,11.13,7.04,12.37,"1,205",5.33,10.48,15.81,10.48,15.81,-
187,양양남대천,양양군(양양대교),24.04.23,3.9,4.4,4.5,5,5.41,5.91,"3,985",0.5,10.41,10.91,10.45,10.95,-
190,삼척오십천,삼척시(상정교),24.04.23,4.1,47.12,5.1,48.12,6.14,49.16,"1,955",43.02,8.29,51.31,8.43,51.45,-
193,강릉남대천,강릉시(회산교),24.04.23,3.8,18.43,4.8,19.43,5.8,20.43,"1,325",14.63,7.74,22.37,8.57,23.2,197


In [ ]:
obsCodeFinal = pd.read_csv('/content/drive/MyDrive/FloodAX/metadata_outputs/obsCodeFinal.csv', sep = ',')
obsCodeFinal.head()

,Unnamed: 0,korObs,korInst,waterElevation,sphereLarge,inOperation,methodObs,codeObs,codeWatershed,korStream,1차,2차,3차,4차,5차,6차,7차,8차
0,0,가평군(가평교),기후에너지환경부,51.62,한강,운영,T/M,1013655.0,101306,가평천하류,143.0,38.0,8.0,3.0,2.0,1.0,NaN,NaN
1,1,가평군(대보교),기후에너지환경부,102.19,한강,운영,T/M,1015620.0,101504,조종천하류,86.0,20.0,5.0,2.0,1.0,NaN,NaN,NaN
2,2,가평군(선촌2교),기후에너지환경부,52.88,한강,운영,T/M,1015636.0,101501,미원천,75.0,19.0,5.0,2.0,2.0,NaN,NaN,NaN
3,3,가평군(신상교),기후에너지환경부,137.63,한강,운영,T/M,1015611.0,101503,조종천상류,77.0,25.0,5.0,2.0,1.0,NaN,NaN,NaN
4,4,가평군(신청평대교),기후에너지환경부,23.93,한강,운영,T/M,1015645.0,101506,청평댐하류,139.0,30.0,6.0,1.0,1.0,NaN,1.0,NaN


In [ ]:
# korObs from obsCodeFinal이랑
# korFloodAlert from obsDesign이랑 동일한지 확인
uniqkorObs = set(obsCodeFinal['korObs'])
uniqkorFloodAlert = set(obsDesign['korFloodAlert(korStream)'])

In [ ]:
uniqkorObs == uniqkorFloodAlert

False

In [ ]:
uniqkorFloodAlert.intersection(uniqkorObs)

{'가평군(가평교)',
 '가평군(대보교)',
 '강릉시(송림교)',
 '강릉시(회산교)',
 '고양시(원당교)',
 '광주시(경안교)',
 '광주시(섬뜰교)',
 '괴산군(목도교)',
 '괴산군(비도교)',
 '남양주시(부평교)',
 '남양주시(왕숙교)',
 '남양주시(진관교)',
 '단양군(단양1교)',
 '동두천시(송천교)',
 '삼척시(상정교)',
 '서울시(너부대교)',
 '서울시(대곡교)',
 '서울시(신대방1교)',
 '서울시(오금교)',
 '서울시(중랑교)',
 '서울시(한강대교)',
 '양양군(양양대교)',
 '양평군(흑천교)',
 '여주시(여주대교)',
 '여주시(원부교)',
 '여주시(율극교)',
 '여주시(흥천대교)',
 '연천군(사랑교)',
 '연천군(신천교)',
 '연천군(임진교)',
 '연천군(차탄교)',
 '영월군(신천교)',
 '영월군(영월대교)',
 '영월군(옥동교)',
 '영월군(주천교)',
 '오산시(탑동대교)',
 '원주시(문막교)',
 '원주시(원주교)',
 '음성군(총천교)',
 '의정부시(신곡교)',
 '인제군(리빙스턴교)',
 '인제군(왕성동교)',
 '정선군(나전교)',
 '정선군(낙동교)',
 '정선군(남평대교)',
 '정선군(송천교)',
 '정선군(와평교)',
 '정선군(정선제1교)',
 '정선군(제1여량교)',
 '제천시(구곡교)',
 '제천시(부수동교)',
 '철원군(장수대교)',
 '충주시(국원대교)',
 '파주시(만장교)',
 '파주시(비룡대교)',
 '평창군(평창교)',
 '평택시(군문교)',
 '평택시(동연교)',
 '평택시(진위1교)',
 '포천시(영평교)',
 '포천시(은현교)',
 '포천시(포천대교)',
 '홍천군(용선교)',
 '홍천군(홍천교)'}

In [ ]:
uniqkorFloodAlert - uniqkorObs

set()

전체 한강홍수통제소(`uniqkorObs`) 관측소 중에서 일부만 홍수특보지점(`uniqkorFloodAlert`)으로 지정해 보고함.

해당 관측소만 계획홍수위(`designFloodLevel`)와 계획홍수량(`designFloodCharge`) 보고함.

-> 다른 관측소랑 비교했을 때 어떤 차이있는지 확인해볼 필요 있을 듯.



In [ ]:
obsDesign.columns

Index(['korStream', 'korFloodAlert(korStream)', 'dateNotice', 'aFLAdvisory(m)',
       'aFLAdvisory(El.m)', 'aFLAlert(m)', 'aFLAlert(El.m)',
       'designFloodLevel(m)', 'designFloodLevel(El.m)',
       'designFloodCharge(m3/sec)', '(El.m)', 'heightEmbarkmentLeft(m)',
       'heightEmbarkmentLeft(El.m)', 'heightEmbarkmentRight(m)',
       'heightEmbarkmentRight(El.m)', 'widthRiver(m)'],
      dtype='object')

In [ ]:
obsDesign.columns = ['korStream', 'korObs', 'dateNotice', 'aFLAdvisory(m)',
       'aFLAdvisory(El.m)', 'aFLAlert(m)', 'aFLAlert(El.m)',
       'designFloodLevel(m)', 'designFloodLevel(El.m)',
       'designFloodCharge(m3/sec)', '(El.m)', 'heightEmbarkmentLeft(m)',
       'heightEmbarkmentLeft(El.m)', 'heightEmbarkmentRight(m)',
       'heightEmbarkmentRight(El.m)', 'widthRiver(m)']

In [ ]:
# 일단 계획홍수위/수량 자료 있는 관측소는 해당 정보 추가
obsFinal = pd.merge(obsCodeFinal, obsDesign, how='left', on='korObs')

In [ ]:
# 계획홍수위/수량 관련 정보 결측은 '-'으로 fill
obsFinal = obsFinal.fillna('-')

In [ ]:
obsDesign.head()

,korStream,korObs,dateNotice,aFLAdvisory(m),aFLAdvisory(El.m),aFLAlert(m),aFLAlert(El.m),designFloodLevel(m),designFloodLevel(El.m),designFloodCharge(m3/sec),(El.m),heightEmbarkmentLeft(m),heightEmbarkmentLeft(El.m),heightEmbarkmentRight(m),heightEmbarkmentRight(El.m),widthRiver(m)
7,한강,서울시(한강대교),00.04.15,8.5,10.57,10.5,12.57,12.8,14.87,"37,000",2.07,15.93,18,16.43,18.5,"1,013"
10,한강,정선군(남평대교),24.04.23,5.5,337.56,6.5,338.56,7.56,339.62,"3,669",332.06,8.88,340.94,8.88,340.94,-
13,한강,여주시(여주대교),12.05.15,6,38.53,8,40.53,9.19,41.72,"16,070",32.53,14.6,47.13,14.59,47.12,502
16,한강,정선군(정선제1교),21.12.28,6.7,301.99,8.6,303.89,10.5,305.79,"4,934",295.29,15.52,310.81,15.5,310.79,340
19,한강,영월군(영월대교),07.04.05,6.6,189.81,8.5,191.71,10.5,193.71,"5,656",183.21,13.8,197.01,13.8,197.01,260


In [ ]:
obsFinal.columns

Index(['Unnamed: 0', 'korObs', 'korInst', 'waterElevation', 'sphereLarge',
       'inOperation', 'methodObs', 'codeObs', 'codeWatershed', 'korStream_x',
       '1차', '2차', '3차', '4차', '5차', '6차', '7차', '8차', 'korStream_y',
       'dateNotice', 'aFLAdvisory(m)', 'aFLAdvisory(El.m)', 'aFLAlert(m)',
       'aFLAlert(El.m)', 'designFloodLevel(m)', 'designFloodLevel(El.m)',
       'designFloodCharge(m3/sec)', '(El.m)', 'heightEmbarkmentLeft(m)',
       'heightEmbarkmentLeft(El.m)', 'heightEmbarkmentRight(m)',
       'heightEmbarkmentRight(El.m)', 'widthRiver(m)'],
      dtype='object')

In [ ]:
obsFinal.head()

,Unnamed: 0,korObs,korInst,waterElevation,sphereLarge,inOperation,methodObs,codeObs,codeWatershed,korStream_x,...,aFLAlert(El.m),designFloodLevel(m),designFloodLevel(El.m),designFloodCharge(m3/sec),(El.m),heightEmbarkmentLeft(m),heightEmbarkmentLeft(El.m),heightEmbarkmentRight(m),heightEmbarkmentRight(El.m),widthRiver(m)
0,0,가평군(가평교),기후에너지환경부,51.62,한강,운영,T/M,1013655.0,101306,가평천하류,...,55.86,6.2,57.06,"2,444",50.86,8.7,59.56,8.91,59.77,140
1,1,가평군(대보교),기후에너지환경부,102.19,한강,운영,T/M,1015620.0,101504,조종천하류,...,105.13,6.42,105.95,"1,464",99.53,8.71,108.24,9.4,108.93,-
2,2,가평군(선촌2교),기후에너지환경부,52.88,한강,운영,T/M,1015636.0,101501,미원천,...,-,-,-,-,-,-,-,-,-,-
3,3,가평군(신상교),기후에너지환경부,137.63,한강,운영,T/M,1015611.0,101503,조종천상류,...,-,-,-,-,-,-,-,-,-,-
4,4,가평군(신청평대교),기후에너지환경부,23.93,한강,운영,T/M,1015645.0,101506,청평댐하류,...,-,-,-,-,-,-,-,-,-,-


In [ ]:
# Unnamed:0, waterElevation (사유: 메타데이터 아님), korStream 삭제 (사유: 중복)
obsFinal = obsFinal.loc[:, ['korObs', 'korInst', 'sphereLarge',
       'inOperation', 'methodObs', 'codeObs', 'codeWatershed', 'korStream_x',
       '1차', '2차', '3차', '4차', '5차', '6차', '7차', '8차',
       'dateNotice', 'aFLAdvisory(m)', 'aFLAdvisory(El.m)', 'aFLAlert(m)',
       'aFLAlert(El.m)', 'designFloodLevel(m)', 'designFloodLevel(El.m)',
       'designFloodCharge(m3/sec)', '(El.m)', 'heightEmbarkmentLeft(m)',
       'heightEmbarkmentLeft(El.m)', 'heightEmbarkmentRight(m)',
       'heightEmbarkmentRight(El.m)', 'widthRiver(m)']]

obsFinalColEngNorm = ['korObs', 'korInst', 'sphereLarge',
       'inOperation', 'methodObs', 'codeObs', 'codeWatershed', 'korStream',
       'stream1', 'stream2', 'stream3', 'stream4', 'stream5', 'stream6', 'stream7', 'stream8',
       'dateNotice', 'aFLAdvisory(m)', 'aFLAdvisory(El.m)', 'aFLAlert(m)',
       'aFLAlert(El.m)', 'designFloodLevel(m)', 'designFloodLevel(El.m)',
       'designFloodCharge(m3/sec)', '(El.m)', 'heightEmbarkmentLeft(m)',
       'heightEmbarkmentLeft(El.m)', 'heightEmbarkmentRight(m)',
       'heightEmbarkmentRight(El.m)', 'widthRiver(m)']

In [ ]:
obsFinal.head()

,korObs,korInst,sphereLarge,inOperation,methodObs,codeObs,codeWatershed,korStream_x,1차,2차,...,aFLAlert(El.m),designFloodLevel(m),designFloodLevel(El.m),designFloodCharge(m3/sec),(El.m),heightEmbarkmentLeft(m),heightEmbarkmentLeft(El.m),heightEmbarkmentRight(m),heightEmbarkmentRight(El.m),widthRiver(m)
0,가평군(가평교),기후에너지환경부,한강,운영,T/M,1013655.0,101306,가평천하류,143.0,38.0,...,55.86,6.2,57.06,"2,444",50.86,8.7,59.56,8.91,59.77,140
1,가평군(대보교),기후에너지환경부,한강,운영,T/M,1015620.0,101504,조종천하류,86.0,20.0,...,105.13,6.42,105.95,"1,464",99.53,8.71,108.24,9.4,108.93,-
2,가평군(선촌2교),기후에너지환경부,한강,운영,T/M,1015636.0,101501,미원천,75.0,19.0,...,-,-,-,-,-,-,-,-,-,-
3,가평군(신상교),기후에너지환경부,한강,운영,T/M,1015611.0,101503,조종천상류,77.0,25.0,...,-,-,-,-,-,-,-,-,-,-
4,가평군(신청평대교),기후에너지환경부,한강,운영,T/M,1015645.0,101506,청평댐하류,139.0,30.0,...,-,-,-,-,-,-,-,-,-,-


In [ ]:
# codeObs dtype 변환 (float -> int)
  # codeObs 정보 누락된 행 기입 후 처리 필요
obsFinal.loc[:, 'codeObs'] = obsFinal.loc[:, 'codeObs'].astype(int)

ValueError: invalid literal for int() with base 10: '-'

In [ ]:
obsFinal.loc[obsFinal.loc[:, 'codeObs'] == '-', :]

,korObs,korInst,sphereLarge,inOperation,methodObs,codeObs,codeWatershed,korStream_x,1차,2차,...,aFLAlert(El.m),designFloodLevel(m),designFloodLevel(El.m),designFloodCharge(m3/sec),(El.m),heightEmbarkmentLeft(m),heightEmbarkmentLeft(El.m),heightEmbarkmentRight(m),heightEmbarkmentRight(El.m),widthRiver(m)
10,강릉시(난곡교),기후에너지환경부,-,-,-,-,320309,-,-,-,...,-,-,-,-,-,-,-,-,-,-
13,강릉시(우정교),기후에너지환경부,-,-,-,-,130201,연곡천,296.0,86.0,...,-,-,-,-,-,-,-,-,-,-
18,고성군(간촌교),기후에너지환경부,-,-,-,-,101103,북천,209.0,42.0,...,-,-,-,-,-,-,-,-,-,-
79,속초시(응골교),기후에너지환경부,-,-,-,-,130105,청초천,141.0,36.0,...,-,-,-,-,-,-,-,-,-,-
137,영월군(주채교),기후에너지환경부,-,-,-,-,100302,옥동천상류,256.0,53.0,...,-,-,-,-,-,-,-,-,-,-


In [ ]:
obsFinal.to_csv('/content/drive/MyDrive/FloodAX/metadata_outputs/obsFinal.csv')

In [ ]:
# codeObs 누락행 처리
obsFinal.loc[obsFinal.loc[:, 'korObs'] == '강릉시(난곡교)', 'codeObs'] = 1302655
obsFinal.loc[obsFinal.loc[:, 'korObs'] == '강릉시(우정교)', 'codeObs'] = 1302666
obsFinal.loc[obsFinal.loc[:, 'korObs'] == '고성군(간촌교)', 'codeObs'] = 1301620
obsFinal.loc[obsFinal.loc[:, 'korObs'] == '속초시(응골교)', 'codeObs'] = 1301665
obsFinal.loc[obsFinal.loc[:, 'korObs'] == '영월군(주채교)', 'codeObs'] = 1003601

In [ ]:
obsFinal.loc[obsFinal.loc[:, 'korObs'] == '강릉시(난곡교)', :]

,korObs,korInst,sphereLarge,inOperation,methodObs,codeObs,codeWatershed,korStream_x,1차,2차,...,aFLAlert(El.m),designFloodLevel(m),designFloodLevel(El.m),designFloodCharge(m3/sec),(El.m),heightEmbarkmentLeft(m),heightEmbarkmentLeft(El.m),heightEmbarkmentRight(m),heightEmbarkmentRight(El.m),widthRiver(m)
10,강릉시(난곡교),기후에너지환경부,-,-,-,1302655,320309,-,-,-,...,-,-,-,-,-,-,-,-,-,-


표준유역코드, 하천차수, 계획홍수위/수량 관련 컬럼값 포함한 최종 메타데이터 -> `obsFinal.csv`

계획홍수위/수량 관련 컬럼이 많기 때문에, 논의 후 일부 drop 필요함.

In [ ]:
# 제원 API 정보와 교차검증
with open('/content/drive/MyDrive/FloodAX/metadata/한강홍수통제소_제원.xml') as f:
  obsAPI = f.read()

In [ ]:
import re
obsAPI = re.split('<Waterlevel>', obsAPI)

In [ ]:
len(obsAPI)

1193

In [ ]:
obsAPI[1]

'\n            <fw> </fw>\n            <wl>1.55</wl>\n            <wlobscd>1001602</wlobscd>\n            <ymdhm>202604171330</ymdhm>\n        </Waterlevel>\n        '

## 하천코드(codeStream) labelling
---

`하천코드`: 하천의 법적 등급과 이름을 구분하는 인덱스**




```또한, 하천에 주키를 부여하는 방법은 다음과 같다.
하천은 수자원관리 종합정보관리시스템에서 전국은 6개 권역으로 구분하고 본류를 기준으로 상류에서 하류로 순차적으로 번호를 부여한다. 번호는 하천코드 구성 권역 및 수계별 번호(2자리)와 하천등급(1자리), 하천번호(4자리)로 구성된다.
하천 = 수계[2] + 등급[1] + 하천번호[4]


다음은 유역에 주키를 부여하는 방법으로, 유역은 대권역, 중권역, 표준유역으로 구분하여 하천상류에서 하류로 일련번호를 부여(Downstream Order)하고, 서쪽에서 동쪽으로, 북쪽에서 남쪽으로 일련번호를 부여한다.
유역 = 대권역[2] + 중권역[2] + 표준유역[2]
이와 같이 종래는 여러 관리기관의 성격에 맞도록 주키(ID)를 부여하고 그 위치에 대한 필드를 추가하여 지형지물의 위치를 표현하였으며, 종래 이러한 지형지물의 위치 표현은 행정구역 명칭을 이용하거나 또는 수치지도 좌표를 이용하는 방법을 들 수 있으며, 좌표를 이용하는 방법이 가장 일반적으로 이용된다.```

In [ ]:
# 표준유역코드와 하천차수 정보로 추정한 유역 역할 / 홍수 반응 속도 결과가
  # 실제 홍수특보 경계 지역이랑 일치하는지 확인할 필요가 있음.

In [ ]:
codeStream = pd.read_csv('/content/drive/MyDrive/FloodAX/metadata/하천코드.csv',
                         encoding='utf8', sep=',')

In [ ]:
codeStream.head()

,수계,하천명,본류,하천코드,하천등급,시도기점,시군구기점,읍면동기점,경계기점,시도종점,시군구종점,읍면동종점,경계종점
0,한강,한강,한강,1000010,국가,강원,정선,여량,송천 합류점,경기,김포,월곳,용강리유도 31m산정부터 남북으로그은직선
1,한강,평창강,한강,1000170,국가,강원,평창,대화,대화천(지방) 합류점,강원,영월,영월,한강(국가) 합류점
2,한강,달천,한강,1000870,국가,충북,청주,미원,계원천(소) 합류점,충북,충주,중앙탑,한강(국가) 합류점
3,한강,섬강,한강,1001330,국가,강원,횡성,횡성,금계천(지방) 합류점,경기,여주,강천,한강(국가) 합류점
4,한강,원주천,한강,1001470,국가,강원,원주,판부,가리파천 (소하천)합류점,강원,원주,호저,섬강(국가) 합류점


In [ ]:
# 컬럼명 변경
  # 수계랑 하천명만
codeStream.columns

Index(['수계', '하천명 ', '본류', '하천코드', '하천등급', '시도기점', '시군구기점', '읍면동기점', '경계기점',
       '시도종점', '시군구종점', '읍면동종점', '경계종점'],
      dtype='object')

In [ ]:
codeStreamEngNorm = ['sphereLarge1', 'korStream', 'sphereLarge2', 'codeStream', 'levelStream', 'provinceStart', 'cityStart', 'streetStart', 'boundaryStart',
       'provinceEnd', 'cityEnd', 'streetEnd', 'boundaryEnd']
codeStream.columns = codeStreamEngNorm

In [ ]:
obsMergedFinal.head()

,korObs,korInst,waterElevation,sphereLarge,inOperation,methodObs,codeObs,codeWatershed,korStream,1차,2차,3차,4차,5차,6차,7차,8차
0,가평군(가평교),기후에너지환경부,51.62,한강,운영,T/M,1013655.0,101306,가평천하류,143.0,38,8,3,2,1,NaN,NaN
1,가평군(대보교),기후에너지환경부,102.19,한강,운영,T/M,1015620.0,101504,조종천하류,86.0,20,5,2,1,NaN,NaN,NaN
2,가평군(선촌2교),기후에너지환경부,52.88,한강,운영,T/M,1015636.0,101501,미원천,75.0,19,5,2,2,NaN,NaN,NaN
3,가평군(신상교),기후에너지환경부,137.63,한강,운영,T/M,1015611.0,101503,조종천상류,77.0,25,5,2,1,NaN,NaN,NaN
4,가평군(신청평대교),기후에너지환경부,23.93,한강,운영,T/M,1015645.0,101506,청평댐하류,139.0,30,6,1,1,NaN,1,NaN


In [ ]:
# 한강홍수통제소 관측소 파일에는 있는데 하천코드 파일에는 없는 하천
uniqStreamHrcf = set(obsMergedFinal['korStream'])
uniqStreamBase = set(codeStream['korStream'])

print(len(list(uniqStreamHrcf-uniqStreamBase)))
print(len(uniqStreamHrcf))

135
194


In [ ]:
pd.merge(obsMergedFinal, codeStream, how='left', on='korStream')

,korObs,korInst,waterElevation,sphereLarge,inOperation,methodObs,codeObs,codeWatershed,korStream,1차,...,codeStream,levelStream,provinceStart,cityStart,streetStart,boundaryStart,provinceEnd,cityEnd,streetEnd,boundaryEnd
0,가평군(가평교),기후에너지환경부,51.62,한강,운영,T/M,1013655.0,101306,가평천하류,143.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,가평군(대보교),기후에너지환경부,102.19,한강,운영,T/M,1015620.0,101504,조종천하류,86.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,가평군(선촌2교),기후에너지환경부,52.88,한강,운영,T/M,1015636.0,101501,미원천,75.0,...,1020910.0,지방,충북,청주,상당구,대신리,충북,청주,상당구,감천(지방) 합류점
3,가평군(선촌2교),기후에너지환경부,52.88,한강,운영,T/M,1015636.0,101501,미원천,75.0,...,1023930.0,지방,경기,가평,설악,묵안리,경기,가평,설악,북한강(국가) 합류점
4,가평군(신상교),기후에너지환경부,137.63,한강,운영,T/M,1015611.0,101503,조종천상류,77.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
291,횡성군(오산교),기후에너지환경부,120.28,한강,운영,T/M,1006630.0,100603,금계천,161.0,...,1021380.0,지방,강원,홍천,동면,노천리 산46 가랫골천(소) 합류점,강원,횡성,공근,섬강(국가) 합류점
292,횡성군(전천교),기후에너지환경부,108.82,한강,운영,T/M,1006660.0,100604,전천,446.0,...,1021410.0,지방,강원,횡성,우천,하궁리 160-6 하궁저수지 지점,강원,횡성,횡성,섬강(국가) 합류점
293,횡성군(전천교),기후에너지환경부,108.82,한강,운영,T/M,1006660.0,100604,전천,446.0,...,1021810.0,지방,경기,이천,율,월포리 1125-3,경기,이천,율면,석원천(지방) 합류점
294,횡성군(청곡교),기후에너지환경부,149.16,한강,운영,T/M,1006628.0,100603,금계천,161.0,...,1021380.0,지방,강원,홍천,동면,노천리 산46 가랫골천(소) 합류점,강원,횡성,공근,섬강(국가) 합류점
